# 1. OLSによるATE推定

この notebook では、もっとも基本的な因果効果推定の例として、**線形回帰モデルのもとで平均処置効果（ATE）を OLS で推定する**流れを確認します。

扱うメッセージは次の 2 点です。

1. 共変量 $X$ が処置 $D$ と相関しているとき、$X$ を無視した単回帰は欠落変数バイアスを生む。
2. 線形モデルが正しく、かつ交絡を適切にコントロールできていれば、重回帰における $D$ の係数は ATE の推定値とみなせる。

この notebook は、後続の notebook で扱うノンパラメトリック推定や DML の出発点です。  
まずは必要なライブラリを読み込みます。


In [6]:
import numpy as np
import statsmodels.api as sm

## データの生成

まず、線形回帰モデルに従う人工データを作ります。ここでは

$$
Y = 0.1 + \theta D + X^\top \beta + \varepsilon
$$

というモデルを仮定します。

各記号の意味は次の通りです。

- $Y$ はアウトカム
- $D \in \{0,1\}$ は二値の処置変数
- $X$ は共変量
- $\theta$ は平均処置効果
- $\varepsilon$ は平均ゼロのノイズ

このモデルでは、$X$ を固定したときの処置効果は常に $\theta$ です。したがって、**線形モデルが正しい**という前提のもとでは、ATE も $\theta$ になります。

今回は真の処置効果を $\theta = 3$ に固定してデータを生成します。あとで推定値がどれくらい 3 に近いかを見れば、推定の成否が分かります。


In [14]:
rng = np.random.default_rng(0)

n = 1000

## 処置効果を3とする
treatment_effect = 3

## 共変量を生成
X_dim = 3
X = rng.normal(0,5,(n, X_dim))

## 処置を生成
xi = rng.uniform(-1,1,X_dim)
prob = 1/(1 + np.exp(-xi.dot(X.T) + rng.normal(0, 1, n)))
rand = rng.uniform(0, 1, n)
D = np.zeros((n, 1))
D[rand < prob, :] = 1

## アウトカムを生成
beta = rng.normal(0,1,X_dim)
epsilon = rng.normal(0,1,n)
Y = 0.1 + treatment_effect * D.T + X.dot(beta) + epsilon
Y = Y.T

ここでは、**重回帰が本当に必要になる状況**をわざと作るために、共変量 $X$ と処置 $D$ が相関するように処置を割り当てています。

直感的には、$X$ が「処置を受けやすさ」にも「アウトカム」にも関係しているので、$X$ は交絡になっています。  
このとき $X$ を無視して $Y$ を $D$ だけで回帰すると、$D$ の係数には

- 真の処置効果
- $X$ を通じた見かけの差

が混ざってしまいます。

まずは、その前提が実際に作れているかどうかを、$D$ と $X$ の相関で確認します。


In [15]:
## 相関の確認
np.corrcoef(D.T[0], X.T[0])

array([[1.        , 0.54465877],
       [0.54465877, 1.        ]])

## OLS による平均処置効果の推定（重回帰）

次に、交絡である $X$ を回帰式に含めた重回帰を行います。推定する式は

$$
Y = \alpha + \theta D + X^\top \beta + \text{error}
$$

です。

このとき、線形モデルが正しく、誤差項と説明変数の条件が満たされていれば、$D$ の係数は処置効果 $\theta$ を表します。  
`statsmodels` の出力では、定数項の次の係数が処置変数 $D$ に対応しています。この notebook ではそれが `x1` として表示されます。

ここで見たいのは、**交絡をコントロールした重回帰では $D$ の係数が真値 3 に近づく**という点です。


In [16]:
## 交絡をコントロールしているモデル
DX = np.concatenate([D, X], axis=1)
DX_reg = sm.add_constant(DX)
long_model = sm.OLS(Y, DX_reg)
res = long_model.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.990
Method:                 Least Squares   F-statistic:                 2.372e+04
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        03:23:28   Log-Likelihood:                -1415.1
No. Observations:                1000   AIC:                             2840.
Df Residuals:                     995   BIC:                             2865.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1483      0.053      2.807      0.0

上の結果では、処置変数 $D$ に対応する係数が真の値 3 にかなり近くなっているはずです。

もちろん有限標本なのでぴったり 3 にはなりませんが、重要なのは

- 係数の大きさが真値に近いこと
- 標準誤差も併せて確認できること
- 交絡を入れたことで、見かけの差ではなく「条件付きの差」を見ていること

です。

この意味で、**線形モデルが正しいなら OLS による重回帰は因果効果推定の自然な出発点**になります。


## OLS による推定（単回帰）

今度は、共変量 $X$ を無視して、$Y$ を $D$ だけで回帰してみます。

推定式は

$$
Y = \alpha + \theta_{\text{naive}} D + \text{error}
$$

です。

このモデルでは、本来 $X^\top \beta$ に入るべき成分が誤差項に押し込まれています。しかも、今回は $D$ と $X$ が相関しているようにデータを作っているので、誤差項と $D$ が相関してしまいます。  
これが、線形回帰でいう**欠落変数バイアス**です。

したがって、この単回帰で得られる $D$ の係数は、一般には処置効果を表しません。


In [17]:
import statsmodels.api as sm

## 交絡をコントロールしていないモデル
D_reg = sm.add_constant(D)
long_model = sm.OLS(Y, D_reg)
res = long_model.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.425
Model:                            OLS   Adj. R-squared:                  0.425
Method:                 Least Squares   F-statistic:                     738.6
Date:                Wed, 15 Apr 2026   Prob (F-statistic):          3.54e-122
Time:                        03:24:44   Log-Likelihood:                -3422.1
No. Observations:                1000   AIC:                             6848.
Df Residuals:                     998   BIC:                             6858.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -5.0270      0.315    -15.975      0.0

結果を見ると、単回帰で得られた $D$ の係数は真値 3 から大きくずれているはずです。

これは「OLS が悪い」というより、**交絡を無視したモデル化が悪い**ということです。  
同じ OLS でも、

- 適切な説明変数を入れた重回帰ではうまくいく
- 交絡を落とした単回帰では大きくずれる

という対比が重要です。

因果推論で強調される「交絡のコントロール」は、この単純な例でもはっきり確認できます。


## $X$ と $D$ に相関がない場合

最後に、処置割り当てが共変量 $X$ と独立な状況を作ります。これは、直感的には**ランダム化実験に近い状況**です。

この場合、$X$ はアウトカムには影響していても、処置の割り当てとは関係しません。すると、$X$ を回帰に入れなくても、$D$ の係数は平均処置効果を回復できます。

これから行うのは次の比較です。

1. $X$ を含む重回帰
2. $X$ を含まない単回帰

この 2 つが、今回はどちらも真値 3 に近いことを確認します。  
まずは本当に $X$ と $D$ がほぼ独立になっているかを相関でチェックします。


In [19]:
# XとDとの間に相関が無ければ、Xを無視しても良い

n = 1000

## 処置効果を5とする
treatment_effect = 3

## 共変量を生成
X_dim = 3
X = rng.normal(0,5,(n, X_dim))

## 処置を生成
rand = rng.uniform(0, 1, n)
D = np.zeros((n, 1))
D[rand < 0.7, :] = 1

## アウトカムを生成
beta = rng.normal(0,1,X_dim)
epsilon = rng.normal(0,1,n)
Y = 0.1 + treatment_effect * D.T + X.dot(beta) + epsilon
Y = Y.T

In [20]:
## 相関の確認
np.corrcoef(D.T[0], X.T[0])

array([[1.        , 0.00733306],
       [0.00733306, 1.        ]])

In [21]:
import statsmodels.api as sm

## 交絡をコントロールしているモデル
DX = np.concatenate([D, X], axis=1)
DX_reg = sm.add_constant(DX)
long_model = sm.OLS(Y, DX_reg)
res = long_model.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.965
Model:                            OLS   Adj. R-squared:                  0.965
Method:                 Least Squares   F-statistic:                     6904.
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        03:26:02   Log-Likelihood:                -1419.2
No. Observations:                1000   AIC:                             2848.
Df Residuals:                     995   BIC:                             2873.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1151      0.059      1.945      0.0

In [22]:
import statsmodels.api as sm

## 交絡をコントロールしていないモデル
D_reg = sm.add_constant(D)
long_model = sm.OLS(Y, D_reg)
res = long_model.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.070
Model:                            OLS   Adj. R-squared:                  0.069
Method:                 Least Squares   F-statistic:                     75.44
Date:                Wed, 15 Apr 2026   Prob (F-statistic):           1.51e-17
Time:                        03:26:05   Log-Likelihood:                -3062.1
No. Observations:                1000   AIC:                             6128.
Df Residuals:                     998   BIC:                             6138.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1718      0.305     -0.563      0.5

## この notebook のまとめ

この notebook で確認したことはとても基本的ですが、後続の内容の土台になります。

- 線形モデルが正しいとき、重回帰における処置変数の係数は ATE と解釈できる。
- ただし、そのためには交絡を表す共変量を回帰式に入れる必要がある。
- 交絡を無視すると、単回帰の係数には欠落変数バイアスが入る。
- 逆に、処置割り当てが共変量と独立なら、単回帰でも ATE を回復できる。

次の notebook では、この「回帰関数を推定して ATE を求める」という考え方を、**線形モデルからノンパラメトリック回帰へ**拡張します。
